In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoImageProcessor, AutoModel
from kaggle_secrets import UserSecretsClient


# ============================================================
# НАСТРОЙКИ
# ============================================================

TRAIN_ROOT = Path(r"/kaggle/input/competitions/dl-lab-5-metric-learning/train/train")
TEST_ROOT  = Path(r"/kaggle/input/competitions/dl-lab-5-metric-learning/test_kaggle/test_kaggle")
INPUT_SUBMISSION_PATH  = Path(r"/kaggle/input/competitions/dl-lab-5-metric-learning/submission.csv")
OUTPUT_SUBMISSION_PATH = Path("submission_arcface_v2.csv")

access_token = UserSecretsClient().get_secret("HF_TOKEN")
MODEL_NAME = "facebook/dinov2-base"
EMBED_DIM  = 768

NUM_EPOCHS  = 10     # больше эпох — лучше сходимость
BATCH_SIZE  = 16
LR          = 1e-5   # меньше lr — обучаем все слои аккуратно
ARC_S       = 64.0
ARC_M       = 0.5

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Размер 336 — больше деталей, dinov2 поддерживает
IMG_SIZE = 336

TTA_TRANSFORMS = [
    T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)]),
    T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE), T.RandomHorizontalFlip(p=1.0)]),
    T.Compose([T.Resize(int(IMG_SIZE * 1.1)), T.CenterCrop(IMG_SIZE)]),
]

TRAIN_TRANSFORM = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

NORMALIZE = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


# ============================================================
# ARCFACE LOSS
# ============================================================

class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, s=64.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        sine   = torch.sqrt((1.0 - cosine ** 2).clamp(0, 1))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return F.cross_entropy(output, labels)


# ============================================================
# ДАТАСЕТ
# ============================================================

class ProductDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.samples   = []
        class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
        self.num_classes = len(class_dirs)
        for class_idx, class_dir in enumerate(class_dirs):
            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTS:
                    self.samples.append((img_path, class_idx))
        print(f"Датасет: {self.num_classes} классов, {len(self.samples)} изображений")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label


# ============================================================
# МОДЕЛЬ
# ============================================================

class DINOv2WithHead(nn.Module):
    def __init__(self, backbone, embed_dim):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return F.normalize(self.head(cls_token), dim=-1)


# ============================================================
# ЗАГРУЗКА МОДЕЛИ
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
torch.cuda.empty_cache()

processor = AutoImageProcessor.from_pretrained(MODEL_NAME, token=access_token)
backbone  = AutoModel.from_pretrained(MODEL_NAME, token=access_token)

# Все слои разморожены — обучаем весь backbone
# fp16 позволяет это сделать без OOM
for param in backbone.parameters():
    param.requires_grad = True

# Gradient checkpointing — экономит память при полном backbone
backbone.gradient_checkpointing_enable()

model = DINOv2WithHead(backbone, EMBED_DIM).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Обучаемых параметров: {trainable:,}")
print("Модель загружена.")


# ============================================================
# ОБУЧЕНИЕ С MIXED PRECISION (fp16)
# ============================================================

train_dataset = ProductDataset(TRAIN_ROOT, TRAIN_TRANSFORM)
train_loader  = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

arcface_loss = ArcFaceLoss(EMBED_DIM, train_dataset.num_classes, ARC_S, ARC_M).to(device)

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": LR},
    {"params": model.head.parameters(),     "lr": LR * 10},
    {"params": arcface_loss.parameters(),   "lr": LR * 10},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS * len(train_loader)
)

# GradScaler для fp16 — автоматически масштабирует градиенты
# чтобы избежать underflow при малых значениях
scaler = torch.cuda.amp.GradScaler()

print(f"\nНачинаю обучение: {NUM_EPOCHS} эпох (fp16, full backbone)...")

for epoch in range(NUM_EPOCHS):
    model.train()
    arcface_loss.train()
    total_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images = images.to(device)
        labels = labels.to(device)

        # autocast — автоматически переводит вычисления в fp16
        with torch.cuda.amp.autocast():
            embeddings = model(images)
            loss = arcface_loss(embeddings, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss = {total_loss / len(train_loader):.4f}")

print("Обучение завершено!")
torch.cuda.empty_cache()


# ============================================================
# ПОИСК ИЗОБРАЖЕНИЙ В ТЕСТЕ
# ============================================================

def find_images_by_filename(root):
    filename_to_paths = defaultdict(list)
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            filename_to_paths[path.name].append(path)
    duplicates = {k: v for k, v in filename_to_paths.items() if len(v) > 1}
    if duplicates:
        raise ValueError(f"Повторяющиеся файлы: {list(duplicates.keys())[:5]}")
    return {name: paths[0] for name, paths in filename_to_paths.items()}


submission = pd.read_csv(INPUT_SUBMISSION_PATH)[["id", "file_1", "file_2"]].copy()
print(f"\nСтрок в submission: {len(submission)}")

filename_to_path = find_images_by_filename(TEST_ROOT.resolve())
needed_files  = set(submission["file_1"].astype(str)) | set(submission["file_2"].astype(str))
missing_files = [n for n in needed_files if n not in filename_to_path]
if missing_files:
    raise FileNotFoundError(f"Не найдено {len(missing_files)} файлов.")

needed_paths = [filename_to_path[n] for n in sorted(needed_files)]
print(f"Изображений для инференса: {len(needed_paths)}")


# ============================================================
# ИНФЕРЕНС С TTA
# ============================================================

def get_embedding(image):
    model.eval()
    all_embs = []
    for transform in TTA_TRANSFORMS:
        aug = transform(image)
        if not isinstance(aug, torch.Tensor):
            aug = T.ToTensor()(aug)
        aug = NORMALIZE(aug).unsqueeze(0).to(device)
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                emb = model(aug).squeeze().cpu().float().numpy()
        all_embs.append(emb)
    mean_emb = np.mean(all_embs, axis=0)
    norm = np.linalg.norm(mean_emb)
    return (mean_emb / norm).astype(np.float32)


embeddings = {}
print("\nСчитаю эмбеддинги с TTA...")

for path in tqdm(needed_paths):
    image = Image.open(path).convert("RGB")
    embeddings[path.name] = get_embedding(image)

print(f"Эмбеддингов посчитано: {len(embeddings)}")


# ============================================================
# SIMILARITY И СОХРАНЕНИЕ
# ============================================================

print("Считаю cosine similarity...")
similarities = [
    float(np.dot(embeddings[str(row.file_1)], embeddings[str(row.file_2)]))
    for row in tqdm(submission.itertuples(index=False), total=len(submission))
]

submission["similarity"] = similarities
OUTPUT_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_SUBMISSION_PATH, index=False, encoding="utf-8-sig")

print(f"\nГотово. Файл сохранён: {OUTPUT_SUBMISSION_PATH}")
display(submission.head())
display(submission["similarity"].describe())